# Relax Structure with NequIP — E(3)-Equivariant Neural Network Potential

This notebook demonstrates structural relaxation using **NequIP**,
an E(3)-equivariant message passing neural network interatomic potential.

NequIP achieves high accuracy with small model sizes by using equivariant
features, making it efficient for materials simulations.

We use the **NequIP-OAM-S** foundation model, trained on OMat24 + sAlex + MPTrj datasets.


## 1. Set Input Parameters
### 1.1. Structure and Relaxation


In [ ]:
FOLDER = "uploads"
STRUCTURE_NAME = "Interface"  # Name of the structure to load from local file

RELAXATION_PARAMETERS = {
    "FMAX": 0.05,
}


## 2. Install Packages


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples|torch|nequip")


In [ ]:
from mat3ra.notebooks_utils.pyodide.packages.torch import apply_all_patches

apply_all_patches(include_nequip=True)


## 3. Load Materials


In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials

structure = load_material_from_folder(FOLDER, STRUCTURE_NAME) or Material.create(
    Materials.get_by_name_first_match(STRUCTURE_NAME))

print(f"Structure: {structure.name}")
print(f"Formula: {structure.formula}")


### 3.1. Visualize Input Structure


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

visualize(structure, repetitions=[1, 1, 1], rotation="-90x")


## 4. Apply Relaxation
### 4.1. Load NequIP Model and Create Calculator


In [ ]:
from mat3ra.notebooks_utils.pyodide.packages.torch import load_nequip_model
from nequip.ase import NequIPCalculator
from nequip.data.transforms import ChemicalSpeciesToAtomTypeMapper, NeighborListTransform

# Load the NequIP-OAM-S model from config + state_dict
nequip_model = load_nequip_model("/drive/packages/models/nequip-oam-s-config-sd.pth")

# Build the ASE calculator with proper transforms
r_max = float(nequip_model.metadata["r_max"])
type_names = nequip_model.metadata["type_names"].split(" ")

calculator = NequIPCalculator(
    model=nequip_model,
    device="cpu",
    transforms=[
        ChemicalSpeciesToAtomTypeMapper(type_names),
        NeighborListTransform(r_max=r_max),
    ],
)

print(f"NequIP-OAM-S calculator ready (r_max={r_max} Å)")


### 4.2. Relax with NequIP


In [ ]:
import plotly.graph_objs as go
from IPython.display import display
from plotly.subplots import make_subplots

from mat3ra.made.tools.convert import to_ase
from ase.optimize import BFGS

ase_structure = to_ase(structure)
ase_structure.calc = calculator
dyn = BFGS(ase_structure)

steps = []
energies = []
forces_max = []

# Store original structure
ase_original_structure = ase_structure.copy()

def log_step():
    e = ase_structure.get_potential_energy()
    f = ase_structure.get_forces()
    fmax = max(((f**2).sum(axis=1) ** 0.5))
    step = dyn.nsteps
    steps.append(step)
    energies.append(e)
    forces_max.append(fmax)
    print(f"  Step {step:3d}: E = {e:12.6f} eV | Fmax = {fmax:.6f} eV/Å")

dyn.attach(log_step)

print(f"Starting relaxation (FMAX = {RELAXATION_PARAMETERS['FMAX']} eV/Å)...")
dyn.run(fmax=RELAXATION_PARAMETERS["FMAX"], steps=200)
ase_final_structure = ase_structure.copy()

# Plot energy and forces convergence
fig = make_subplots(rows=1, cols=2, subplot_titles=("Energy", "Max Force"))
fig.add_trace(go.Scatter(x=steps, y=energies, mode="lines+markers", name="Energy"), row=1, col=1)
fig.add_trace(go.Scatter(x=steps, y=forces_max, mode="lines+markers", name="Fmax"), row=1, col=2)
fig.add_hline(y=RELAXATION_PARAMETERS["FMAX"], line_dash="dash", line_color="red", row=1, col=2)
fig.update_xaxes(title_text="Step", row=1, col=1)
fig.update_xaxes(title_text="Step", row=1, col=2)
fig.update_yaxes(title_text="Energy (eV)", row=1, col=1)
fig.update_yaxes(title_text="Fmax (eV/Å)", row=1, col=2)
fig.update_layout(height=400, showlegend=False)
display(fig)

print(f"\nRelaxation converged in {len(steps)} steps")
print(f"Final energy: {energies[-1]:.6f} eV")
print(f"Final Fmax:   {forces_max[-1]:.6f} eV/Å")


## 5. Analyze Results
### 5.1. View Structure Before and After Relaxation


In [ ]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_structure))
material_relaxed = Material.create(from_ase(ase_final_structure))
material_original.name = structure.name
material_relaxed.name = structure.name + " (NequIP Relaxed)"

visualize([material_original, material_relaxed], repetitions=[1, 1, 1], rotation="-90x")


### 5.2. Output interlayer distance before and after relaxation


In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

SUBSTRATE_TAG = 0
FILM_TAG = 1

print(
    f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, SUBSTRATE_TAG, FILM_TAG):.4f} Å")
print(
    f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, SUBSTRATE_TAG, FILM_TAG):.4f} Å")


## References

[1] NequIP: https://github.com/mir-group/nequip

[2] Simon Batzner et al., "E(3)-equivariant graph neural networks for data-efficient and accurate interatomic potentials," Nature Communications (2022)

[3] NequIP-OAM Foundation Models: https://zenodo.org/records/18775904
